In [1]:
# ============================================
# 1. LOAD RAW DATA (BRONZE LAYER)
# ============================================

from pyspark.sql.functions import *

# Load customers CSV
df_customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Bronze/customer/customers.csv")
)

# Load policies CSV
df_policies = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Bronze/policy/policies.csv")
)

# Load claims CSV
df_claims = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("Files/Bronze/claim/claims.csv")
)

# Verify the data
print("Customers:", df_customers.count(), "rows")
print("Policies:", df_policies.count(), "rows")
print("Claims:", df_claims.count(), "rows")

# Display sample records
display(df_customers.limit(10))
display(df_policies.limit(10))
display(df_claims.limit(10))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 3, Finished, Available, Finished, False)

Customers: 50 rows
Policies: 50 rows
Claims: 50 rows


SynapseWidget(Synapse.DataFrame, c94b996c-7a02-43ba-a967-3fc3b1f944ff)

SynapseWidget(Synapse.DataFrame, cb43843c-8ff6-42c4-9d91-e17b0f5686ff)

SynapseWidget(Synapse.DataFrame, de7e1d3a-7f47-4faa-9239-c14ab68713eb)

In [2]:
display(df_customers.limit(5))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b7ca58c1-cca6-4140-bf30-d2aa7b177a00)

## Cleaning the customer data

In [5]:
display(df_customers.limit(5))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9ddbd2d-4ac3-4a35-8e1f-30c2d3e1feb2)

In [6]:
df_customers_clean = (
    df_customers
    .dropna(subset=["cust_id"])
    .withColumn("cust_id", col("cust_id").cast("int"))
    .withColumn("CustomerName", initcap(trim(col("Full Name"))))
    .withColumn("Date_of_Birth", to_date(col("Date_of_Birth")))
    .withColumn("Contact", when(col("Contact").isNull(), "Not Provided").otherwise(col("Contact")))
    .drop("Full Name")
    .dropDuplicates()
)
display(df_customers_clean)

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 94ec2097-1655-4e96-b749-8470aad2591f)

## Cleaning the policy data

In [7]:
display(df_policies.limit(10))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe86437f-b390-41a6-b043-d0f279c2254c)

In [8]:
df_policies_clean = (
    df_policies
    .withColumn("policy_Type", initcap(trim(col("policy_Type"))))
    .withColumn("status", initcap(trim(col("status"))))
    .withColumn("Start_Date", to_date(col("Start_Date"), "dd/MM/yyyy"))
    .withColumn("Coverage_Amount", when(col("Coverage_Amount") == "not available", None)
                .otherwise(col("Coverage_Amount").cast("double")))
    .withColumn("cust_id", col("cust_id").cast("int"))
    .dropna(subset=["cust_id", "policy_id"])
)
display(df_policies_clean)

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bea8f4bd-6cce-4ee7-bbe8-c99e792be3c7)

## Cleaning the Claim data


In [9]:

display(df_claims.limit(5))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fb67acbd-4544-4657-b374-242eae0e18f4)

In [10]:
df_claims_clean = (
    df_claims
    .withColumn("claim_date", to_date(col("claim_date")))
    .withColumn("claim_amount", col("claim_amount").cast("double"))
    .withColumn("status", initcap(trim(col("status"))))
    .dropna(subset=["policy_id"])
)
display(df_claims_clean)


StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7178f68b-c9df-487e-874e-ba0c8af23851)

## Save Cleaned Data as Silver Delta Tables

In [11]:
# ============================================
# 5. CREATE DELTA TABLES (SILVER LAYER)
# ============================================

df_customers_clean.write.mode("overwrite").format("delta").saveAsTable("silver_customers")

df_policies_clean.write.mode("overwrite").format("delta").saveAsTable("silver_policies")

df_claims_clean.write.mode("overwrite").format("delta").saveAsTable("silver_claims")

print("Silver Delta tables created successfully!")

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 13, Finished, Available, Finished, False)

Silver Delta tables created successfully!


In [14]:
%%sql
SELECT *
FROM silver_customers;

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 16, Finished, Available, Finished, False)

<Spark SQL result set with 50 rows and 5 fields>

## Join Silver Tables for Gold Layer Transformation

In [16]:
print("Claims:", df_claims_clean.columns)
print("Policies:", df_policies_clean.columns)
print("Customers:", df_customers_clean.columns)

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 18, Finished, Available, Finished, False)

Claims: ['claim_id', 'policy_id', 'claim_date', 'claim_amount', 'status']
Policies: ['policy_id', 'cust_id', 'policy_Type', 'status', 'Start_Date', 'Coverage_Amount']
Customers: ['cust_id', 'Date_of_Birth', 'Gender', 'Contact', 'CustomerName']


In [19]:
df_joined = (
    df_claims_clean.alias("c")
    .join(
        df_policies_clean.alias("p"),
        col("c.policy_id") == col("p.policy_id"),
        "inner"
    )
    .join(
        df_customers_clean.alias("cu"),
        col("p.cust_id") == col("cu.cust_id"),
        "inner"
    )
    .select(
        col("c.claim_id"),
        col("c.policy_id"),
        col("c.claim_date"),
        col("c.claim_amount"),
        col("c.status").alias("claim_status"),
        col("p.cust_id"),
        col("p.policy_Type"),
        col("p.status").alias("policy_status"),
        col("p.Start_Date"),
        col("p.Coverage_Amount"),
        col("cu.CustomerName"),
        col("cu.Gender"),
        col("cu.Date_of_Birth"),
        col("cu.Contact")
    )
)

display(df_joined.limit(5))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 89b8ef29-3e19-4e00-a626-af6fae326466)

## Creating Gold Aggregation Table

In [22]:
# ============================================
# CREATE GOLD AGGREGATION TABLE
# ============================================

df_aggregated = (
    df_joined
    .groupBy("cust_id", "CustomerName", "Gender")
    .agg(
        count("claim_id").alias("TotalClaims"),

        count(
            when(lower(col("claim_status")) == "approved", 1)
        ).alias("ApprovedClaims"),

        count(
            when(lower(col("claim_status")) == "rejected", 1)
        ).alias("RejectedClaims"),

        round(sum("claim_amount"), 2).alias("TotalClaimAmount"),

        round(avg("claim_amount"), 2).alias("AvgClaimAmount"),

        collect_set(
            lower(col("policy_Type"))
        ).alias("PolicyTypesArray"),

        round(sum("Coverage_Amount"), 2).alias("TotalCoverageAmount"),

        min("claim_date").alias("FirstClaimDate"),

        max("claim_date").alias("LastClaimDate")
    )
    .withColumn(
        "PolicyTypes",
        concat_ws(", ", col("PolicyTypesArray"))
    )
    .withColumn(
        "ClaimToCoverageRatio",
        when(
            col("TotalCoverageAmount") != 0,
            round(
                (col("TotalClaimAmount") / col("TotalCoverageAmount")) * 100,
                2
            )
        )
    )
    .drop("PolicyTypesArray")
)

display(df_aggregated)

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 440fd955-adc7-4708-ae09-d8c0eb5e1b5c)

## Saving the Gold Table

In [23]:
# ============================================
# SAVE GOLD DELTA TABLE
# ============================================

df_aggregated.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("insurance_aggrgate")

print("Gold table created successfully!")

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 25, Finished, Available, Finished, False)

Gold table created successfully!


## Save Aggregated Data as Gold Delta Files

In [24]:
# SAVE THE CLEANED AND ENRICHED GOLD OUTPUT

df_aggregated.write \
    .mode("overwrite") \
    .format("delta") \
    .save("abfss://61d637b0-f8d1-41d9-b440-aeb800d1186c@onelake.dfs.fabric.microsoft.com/238a1ee2-270e-48e4-b4fc-7c4e8122980c/Files/Gold")

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 26, Finished, Available, Finished, False)

In [25]:
%%sql
SELECT *
FROM insurance_aggrgate;

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 27, Finished, Available, Finished, False)

<Spark SQL result set with 24 rows and 13 fields>

## Top 10 High-Value Customers by Claim Amount

In [26]:
display(spark.sql("""
SELECT CustomerName, TotalClaimAmount
FROM insurance_aggrgate
ORDER BY TotalClaimAmount DESC
LIMIT 10
"""))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ba3d251-acc8-4ee1-ba60-86fd289c89d3)

## Claim Approval Rate by Gender

In [27]:
display(spark.sql("""
SELECT
    Gender,
    SUM(ApprovedClaims) AS ApprovedClaims,
    SUM(RejectedClaims) AS RejectedClaims,
    SUM(TotalClaims) AS TotalClaims,
    ROUND(
        SUM(ApprovedClaims) * 100.0 / SUM(TotalClaims),
        2
    ) AS ApprovalRate
FROM insurance_aggrgate
GROUP BY Gender
ORDER BY ApprovalRate DESC
"""))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 29, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4cd6533c-a52c-45d4-9f34-a7772b229ade)

## Claim to Coverage Ratio Distribution

In [29]:
display(spark.sql("""
SELECT
    CustomerName,
    ClaimToCoverageRatio
FROM insurance_aggrgate
WHERE ClaimToCoverageRatio IS NOT NULL
ORDER BY ClaimToCoverageRatio DESC
"""))

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 31, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d0567036-9002-4a54-8cef-44c47716e0f3)

## Claims by Policy Types (Explode for Analysis)

In [30]:
from pyspark.sql.functions import explode, split, col, count

df_policy_split = (
    df_aggregated
    .withColumn("PolicyType", explode(split(col("PolicyTypes"), ",\\s*")))
    .groupBy("PolicyType")
    .agg(count("*").alias("CustomerCount"))
)

display(df_policy_split)

StatementMeta(, b2b1f6db-ac38-41c0-95d9-0506e90ca6dd, 32, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1bf01009-43fc-4231-b43e-9c4f8448273f)